## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

pd.set_option('mode.chained_assignment', None)

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)


## 2. Load Preprocessed Data


In [49]:
# Load preprocessed data from Person 5
X_train = pd.read_csv('data/processed/X_train_diabetes.csv')
X_test = pd.read_csv('data/processed/X_test_diabetes.csv')
y_train = pd.read_csv('data/processed/y_train_diabetes.csv').values.flatten()
y_test = pd.read_csv('data/processed/y_test_diabetes.csv').values.flatten()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (800, 8)
X_test shape: (154, 8)
y_train shape: (800,)
y_test shape: (154,)


## 3. Model 1: Logistic Regression


In [ ]:
# Hyperparameter tuning for Logistic Regression (using F1 score for imbalanced data)
print("Tuning Logistic Regression...")
lr_param_dist = {
    'C': [0.5, 1, 2, 5, 10, 20],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'class_weight': ['balanced']  # Focus on balanced for imbalanced data
}
lr_base = LogisticRegression(random_state=42, max_iter=2000)
# Use F1 score instead of accuracy for better performance on imbalanced data
lr_grid = RandomizedSearchCV(lr_base, lr_param_dist, n_iter=12, cv=5, 
                             scoring='f1', n_jobs=-1, verbose=0, random_state=42)
lr_grid.fit(X_train, y_train)

lr_model = lr_grid.best_estimator_
y_pred_lr = lr_model.predict(X_test)
y_proba_lr = lr_model.predict_proba(X_test)

# Save predictions and probabilities to evaluate
os.makedirs('data/processed', exist_ok=True)
pd.DataFrame(y_pred_lr).to_csv('data/processed/y_pred_lr.csv', index=False)
pd.DataFrame(y_proba_lr).to_csv('data/processed/y_proba_lr.csv', index=False)

# Save the model
joblib.dump(lr_model, 'models/logistic_regression.pkl')
print(f"Logistic Regression - Best params: {lr_grid.best_params_}")
print(f"Logistic Regression - Best CV score: {lr_grid.best_score_:.4f}")
print("Logistic Regression - Training completed. Predictions and model saved.")


Tuning Logistic Regression...
Logistic Regression - Best params: {'solver': 'lbfgs', 'penalty': 'l2', 'class_weight': 'balanced', 'C': 0.5}
Logistic Regression - Best CV score: 0.7321
Logistic Regression - Training completed. Predictions and probabilities saved.


## 4. Model 2: Naive Bayes


In [ ]:
# Hyperparameter tuning for Naive Bayes (using F1 score)
print("Tuning Naive Bayes...")
nb_param_grid = {
    'var_smoothing': [1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]
}
nb_base = GaussianNB()
# Use F1 score for better performance on imbalanced data
nb_grid = GridSearchCV(nb_base, nb_param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0)
nb_grid.fit(X_train, y_train)

nb_model = nb_grid.best_estimator_
y_pred_nb = nb_model.predict(X_test)
y_proba_nb = nb_model.predict_proba(X_test)

# Save predictions to evaluate
pd.DataFrame(y_pred_nb).to_csv('data/processed/y_pred_nb.csv', index=False)
pd.DataFrame(y_proba_nb).to_csv('data/processed/y_proba_nb.csv', index=False)

# Save the model
joblib.dump(nb_model, 'models/naive_bayes.pkl')
print(f"Naive Bayes - Best params: {nb_grid.best_params_}")
print(f"Naive Bayes - Best CV score: {nb_grid.best_score_:.4f}")
print("Naive Bayes - Training completed. Predictions and model saved.")


Tuning Naive Bayes...
Naive Bayes - Best params: {'var_smoothing': 1e-10}
Naive Bayes - Best CV score: 0.7478
Naive Bayes - Training completed. Predictions and probabilities saved.


## 5. Model 3: Decision Tree


In [ ]:
# Hyperparameter tuning for Decision Tree (using F1 score)
print("Tuning Decision Tree...")
dt_param_dist = {
    'max_depth': [5, 7, 10, 12, 15],
    'min_samples_split': [5, 10, 15, 20],
    'min_samples_leaf': [2, 4, 6, 8],
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']  # Focus on balanced for imbalanced data
}
dt_base = DecisionTreeClassifier(random_state=42)
# Use F1 score and more conservative parameters to prevent overfitting
dt_grid = RandomizedSearchCV(dt_base, dt_param_dist, n_iter=30, cv=5, 
                             scoring='f1', n_jobs=-1, verbose=0, random_state=42)
dt_grid.fit(X_train, y_train)

dt_model = dt_grid.best_estimator_
y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)

# Save predictions to evaluate
pd.DataFrame(y_pred_dt).to_csv('data/processed/y_pred_dt.csv', index=False)
pd.DataFrame(y_proba_dt).to_csv('data/processed/y_proba_dt.csv', index=False)

# Save the model
joblib.dump(dt_model, 'models/decision_tree.pkl')
print(f"Decision Tree - Best params: {dt_grid.best_params_}")
print(f"Decision Tree - Best CV score: {dt_grid.best_score_:.4f}")
print("Decision Tree - Training completed. Predictions and model saved.")


Tuning Decision Tree...
Decision Tree - Best params: {'min_samples_split': 20, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 7, 'criterion': 'gini', 'class_weight': 'balanced'}
Decision Tree - Best CV score: 0.7598
Decision Tree - Training completed. Predictions and probabilities saved.


## 6. Model 4: SVM


In [ ]:
# Hyperparameter tuning for SVM (using F1 score)
print("Tuning SVM (this may take a moment)...")
svm_param_dist = {
    'C': [1, 2, 5, 10, 20],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'class_weight': ['balanced']  # Focus on balanced for imbalanced data
}
svm_base = SVC(random_state=42, probability=True)
# Use F1 score for better performance on imbalanced data
svm_grid = RandomizedSearchCV(svm_base, svm_param_dist, n_iter=20, cv=5, 
                              scoring='f1', n_jobs=-1, verbose=1, random_state=42)
svm_grid.fit(X_train, y_train)

svm_model = svm_grid.best_estimator_
y_pred_svm = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)

# Save predictions to evaluate
pd.DataFrame(y_pred_svm).to_csv('data/processed/y_pred_svm.csv', index=False)
pd.DataFrame(y_proba_svm).to_csv('data/processed/y_proba_svm.csv', index=False)

# Save the model
joblib.dump(svm_model, 'models/svm.pkl')
print(f"SVM - Best params: {svm_grid.best_params_}")
print(f"SVM - Best CV score: {svm_grid.best_score_:.4f}")
print("SVM - Training completed. Predictions and model saved.")


Tuning SVM (this may take a moment)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
SVM - Best params: {'kernel': 'rbf', 'gamma': 'scale', 'class_weight': 'balanced', 'C': 20}
SVM - Best CV score: 0.8133
SVM - Training completed. Predictions and probabilities saved.


## 7. Model Comparison (Without Metrics)

**Note**: Evaluation metrics (Accuracy, F1, Confusion Matrix) in file 01_data_plaplapla


In [54]:
# Simple comparison table (without metrics)
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree', 'SVM'],
    'Status': ['Trained', 'Trained', 'Trained', 'Trained'],
    'Predictions Saved': ['Yes', 'Yes', 'Yes', 'Yes']
})

display(comparison)


,Model,Status,Predictions Saved
0,Logistic Regression,Trained,Yes
1,Naive Bayes,Trained,Yes
2,Decision Tree,Trained,Yes
3,SVM,Trained,Yes
